## CNN Convolutional Neural Network

In [22]:
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

import numpy as np

#### simple guide

```python
nn.Conv2d(
    in_channel=1,   #gray   RGB 彩色图像通常有 3 个通道
    out_channels=16,    # 表示这一层使用 16 个不同的卷积核，生成 16 张特征图
                        # 假设输入：[Batch Size, 1, 8, 8]
                        # 经过：`nn.Conv2d(1, 16, kernel_size=3, padding=1)`
                        # 输出：[Batch Size, 16, 8, 8]
    kernel_size=3,
    padding=1,  #Padding 就是在图像周围填充一些数值，通常填充 0
    stride=1    #how many pixel the kernel moves once
)

为什么设置 padding=1？

因为使用 3×3 卷积核，且步长为 1 时，如果没有 Padding，图像尺寸会缩小。

原始：
8 × 8

卷积后：
6 × 6

设置 padding=1 后：
8 × 8

因此，这里 Padding 的作用就是保持空间尺寸。

```python
nn.MaxPool2d(kernel_size=2)     #MaxPool2d 最大池化

除了卷积，CNN 中还有一个重要操作：
nn.MaxPool2d(kernel_size=2)

它的作用是降低特征图的空间尺寸。

例如输入：

1  3  2  4

5  6  1  2

7  2  8  3

4  1  6  9

使用 2×2 的最大池化：每个 2×2 区域只保留最大的数字。

输出：

6  4

7  9

空间尺寸从：
4 × 4

变成：
2 × 2

为什么需要池化？

主要原因：

    减少特征图尺寸。
    降低后续计算量。
    保留局部区域中比较显著的特征。
    提高模型对小范围位置变化的适应能力。

注意：Max Pooling 不是卷积，它没有需要通过反向传播学习的卷积核参数。

## Prepare Data

the data type for `nn,Conv2d` is `[batch_size, in_channel, heights, width]`

In [3]:
# loading hand written digits using `load_digits()`
digits=load_digits()

X=digits.images.astype(np.float32)/16.0

# Add channel dimension
print("origin X:",X.shape)
X=X[:,None,:,:] #NumPy array operation, `None` means adding a new dimension 

y=digits.target.astype(np.int64)

print(X.shape)
print(y.shape)


origin X: (1797, 8, 8)
(1797, 1, 8, 8)
(1797,)


In [4]:
#splitting the datasets
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y  #`stratify分层` it means stratifying samples by the proportion of y 
    #to keep the category distribution in the training data and the test data as consistent as posible
) 

In [5]:
#Creating Dataset
train_dataset=TensorDataset(torch.from_numpy(X_train),torch.from_numpy(y_train))
test_dataset=TensorDataset(torch.from_numpy(X_test),torch.from_numpy(y_test))

In [6]:
train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
test_dataloader=DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

## Define CNN

In [7]:
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()

        #Feature Extraction
        self.features=nn.Sequential(
            #Input: [batch_size,1,8,8]

            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2),    # [batch_size,16,8,8] -> [batch_size,16,4,4]

            nn.Conv2d(             #[batch_size,32,4,4] -> [batch_size,32,4,4]
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2)  #[batch_size,32,4,4] -> [batch_size,32,2,2]
        )

        #Classification
        self.classifier=nn.Sequential(
            nn.Flatten(),   #dim=0 stay still, and multiply the other dimensions to dim=1
            nn.Linear(32*2*2,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )

    def forward(self,x):
        x=self.features(x)
        x=self.classifier(x)
        return x

## Training model

In [8]:
# using `torch.cuda.is_available()` to check the current environment
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [9]:
model=DigitCNN().to(device)

In [10]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)

In [ ]:
epochs=30

for epoch in range(1,epochs+1):
    model.train()

    train_loss_sum=0.0
    train_correct=0
    train_total=0

    for feature, labels in train_loader:

        feature=feature.to(device)
        labels=labels.to(device)

        optimizer.zero_grad()

        logits=model(feature)   #`model.__call__(features)` underneath. And in PyTorch, __call__ will execute `forward()` and other necessary mechanism like `hook`

        loss=loss_fn(logits, labels)

        loss.backward()

        optimizer.step()

        batch_size=labels.size(0)
        train_loss_sum+=loss.item()*batch_size
        train_correct+=(logits.argmax(dim=1)==labels).sum().item()  #the output of `logits.argmax(1)==label` is tensor([ True, True, False])
        train_total+=batch_size

## Evaluate

In [16]:
def evaluate(model:nn.Module, loader: DataLoader, loss_fn: nn.Module, device:torch.device):
    model.eval()

    total_loss=0.0
    correct=0
    total=0

    with torch.no_grad():
        for features, labels in loader:
            features=features.to(device)
            labels=labels.to(device)

            logits=model(features)
            loss=loss_fn(logits,labels)

            batch_size=labels.size(0)
            total_loss+=loss.item()*batch_size
            correct+=(logits.argmax(dim=1)==labels).sum().item()
            total+=batch_size

    average_loss=total_loss/total
    accuracy=correct/total

    return average_loss, accuracy

In [17]:
test_loss,test_accuracy=evaluate(model,test_dataloader,loss_fn,device)
print(test_loss)
print(test_accuracy)

0.08715860214498308
0.9777777777777777


## Check the prediction

In [19]:
model.eval()

sample_x,sample_y=next(iter(test_dataloader))

with torch.no_grad():
    predictions=model(sample_x.to(device)).argmax(dim=1).cpu()

for true_label, pred in zip(sample_y[:12].tolist(), predictions[:12].tolist()):
    mark = "✓" if true_label == pred else "✗"
    print(f"  {true_label} -> {pred} {mark}")

  5 -> 5 ✓
  7 -> 7 ✓
  9 -> 9 ✓
  4 -> 4 ✓
  2 -> 2 ✓
  7 -> 7 ✓
  5 -> 5 ✓
  0 -> 0 ✓
  9 -> 9 ✓
  7 -> 7 ✓
  4 -> 4 ✓
  0 -> 0 ✓


## Save the trained model parameters

In [23]:
# using `model.state_dict()` to get the model parameters, and using `torch.save()` to save to digit_mlp.pth
# note: digit_mlp.pth only saves the parameters, not the model structure. We need to redefine the netwoek structure
model_path=Path("digit_mlp.pth")

torch.save(model.state_dict(),model_path)